### Load Document

In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "../data/knowledge/nk_securities/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress = False
)

documents = loader.load()
print(len(documents))

/home/sanchit125/RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4


### Chunking

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def text_splitter (docuemnts: list[Document], chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "", " ", "\n"]
    )
    split_docs = splitter.split_documents(docuemnts)

    return split_docs

chunks = text_splitter(documents)
print(len(chunks))

14


### Embedding

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

class EmbeddingManager:
    def __init__(self) -> None:
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
        )

    def embed_document (self, chunks: list[Document]) -> list[list[float]]:
        texts = [chunk.page_content for chunk in chunks]

        embeddings = self.embedding_model.embed_documents(texts)

        return embeddings

    def embed_query (self, query: str) -> list[float]:
        embedding = self.embedding_model.embed_query(query)
        return embedding

embedding_manager = EmbeddingManager()

chunk_embedding = embedding_manager.embed_document(chunks)
print(chunk_embedding)

/home/sanchit125/RAG/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 443.57it/s]


[[-0.044086456298828125, 0.018120061606168747, -0.05722677707672119, 0.0017979864496737719, -0.02861248143017292, -0.07563814520835876, 0.023432081565260887, 0.019095223397016525, -0.010538680478930473, 0.03595900163054466, -0.07555478811264038, 0.01378690730780363, 0.018496152013540268, -0.02177700586616993, 0.0016124555841088295, -0.0043595051392912865, 0.012829677201807499, -0.019007539376616478, -0.02454044483602047, -0.15790538489818573, -0.07116103917360306, -0.08281707018613815, 0.0547107569873333, -0.045584648847579956, 0.02402811497449875, 0.006355596240609884, 0.006789420265704393, -0.011493399739265442, 0.008318883366882801, -0.041687194257974625, -0.003658013418316841, -0.00041654412052594125, 0.037693656980991364, 0.0951405018568039, 0.009495842270553112, 0.13524578511714935, -0.0393618680536747, 0.035004302859306335, 0.008078115060925484, -0.025519751012325287, -0.016266800463199615, -0.04022728279232979, -0.00585961015895009, -0.04419156163930893, 0.07494226098060608, -0

### Vector Store

In [5]:
from chromadb import PersistentClient
from langchain_core.documents import Document
from typing import cast
from chromadb.api.types import Metadata, Embedding

class VectorStore:
    def __init__(self, collection_name: str = "nk_securities", persist_directory: str = "../data/vector_store/nk_securities") -> None:
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = PersistentClient(
            path=self.persist_directory
        )
        self.connection = self.client.get_or_create_collection(
            name=self.collection_name
        )

    def add_document(self, chunks: list[Document], embeddings: list[list[float]]):
        ids = [f"{i}" for i in range(len(chunks))]
        metadatas = [chunk.metadata for chunk in chunks]
        texts = [chunk.page_content for chunk in chunks]

        self.connection.add(
            ids=ids,
            documents=texts,
            metadatas=cast(list[Metadata], metadatas),
            embeddings=cast(list[Embedding], embeddings)
        )

    def similarity_search (self, query_embedding: list[float], k: int=5):
        results = self.connection.query(
            query_embeddings = cast(list[Embedding], query_embedding),
            n_results = k
        )

        documents = results["documents"]
        metadatas = results["metadatas"]

        if not documents:
            return []

        if not metadatas:
            return [[] for _ in documents]

        return [
            Document(
                page_content = text,
                metadata = meta or {}
            )
            for text, meta in zip(documents[0], metadatas[0])
        ]

vectors_store = VectorStore()
vectors_store.add_document(chunks, chunk_embedding)

### Retrieval

In [6]:
query = "What is the best match in the resume with the NK securities JD"

query_embeddings = embedding_manager.embed_query(query)

result = vectors_store.similarity_search(query_embeddings, 5)
print(result)

[Document(metadata={'creator': '', 'author': '', 'title': 'Software Developer (Platform/AI) Internship_Job Description.docx', 'producer': 'Skia/PDF m153 Google Docs Renderer', 'format': 'PDF 1.4', 'source': '../data/knowledge/nk_securities/Software Developer (Platform_AI) Internship_Job Description.docx (3).pdf', 'file_path': '../data/knowledge/nk_securities/Software Developer (Platform_AI) Internship_Job Description.docx (3).pdf', 'trapped': '', 'subject': '', 'modDate': '', 'page': 0, 'moddate': '', 'keywords': '', 'creationDate': '', 'creationdate': '', 'total_pages': 2}, page_content='NK Securities Research \nSoftware Developer (Platform/AI) Internship (6 Months) \nLocation: Gurugram \nNK Securities Research is a leading financial firm that leverages cutting edge technology and \nsophisticated algorithms to trade the financial markets. Founded in 2011, we have gained invaluable \nexperience in the field of High Frequency Trading across different asset classes. \n \nKey Responsibili

### Using FAISS vector store

In [7]:
from langchain_community.vectorstores import FAISS

class FaissStore:
    def __init__(self, embedding_model) -> None:
        self.embedding_model = embedding_model
        self.vector_store = None

    def add_document(self, chunks: list[Document]):
        self.vector_store = FAISS.from_documents(
            chunks, self.embedding_model
        )

    def semantic_query(self, query: str):
        if self.vector_store is None:
            return []
        return self.vector_store.similarity_search(
            query,
            5
        )

faiss_store = FaissStore(embedding_manager.embedding_model)
faiss_store.add_document(chunks)

result = faiss_store.semantic_query("How are you")
print(result)


[Document(id='5631c8d4-91f3-4ce5-b0b9-1ff449a71781', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-22T14:30:33+05:30', 'source': '../data/knowledge/nk_securities/110123099_sanchitRathore.pdf', 'file_path': '../data/knowledge/nk_securities/110123099_sanchitRathore.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Charvini Doddi', 'subject': '', 'keywords': '', 'moddate': '2026-08-22T14:30:33+05:30', 'trapped': '', 'modDate': "D:20260822143033+05'30'", 'creationDate': "D:20260822143033+05'30'", 'page': 0}, page_content='tomated \njudging against predefined database test cases to generate Accepted / Wrong Answer verdicts. \nYear \nDegree/Examination \nInstitution/Board \nCGPA/Percentage \n2023-Present \nB.Tech- ICE \nNIT, Trichy \n8.98 \n2022 \nClass XII \nMahar Regiment Public School, \nSagar, Madhya Pradesh, CBSE \n93.80% \n2020 \nClass X \nMahar Regiment Public School, \nSagar Madhy